# 实验三 · QQ 邮箱采集（Playwright）
陈泓宇 · 数据1241 · 112411160433

在课程 Docker 的 JupyterLab 中按顺序运行。手机扫码授权后，程序读取收件箱列表和详情，将原始数据保存在本地。公开结果只包含代号、正文长度与核对统计。优先使用课程练习邮箱。

## 第一阶段：扫码登录（第1—2课时）

### 单元1：准备浏览器

In [14]:
import asyncio
import re
import sys
from pathlib import Path
from importlib.metadata import version

import pandas as pd
from playwright.async_api import async_playwright, TimeoutError as BrowserTimeout

PRIVATE = Path.cwd() / "private_data"
PUBLIC = Path.cwd() / "data"
PRIVATE.mkdir(exist_ok=True)
sys.path.insert(0, str(Path.cwd().parent / "L07_L08_visual"))
import course_visual
course_visual.prepare()

reuse = (all(name in globals() for name in ("p", "browser", "context", "page"))
         and browser.is_connected() and not page.is_closed()
         and page in context.pages)
if not reuse:
    p = await async_playwright().start()
    browser = await p.chromium.launch(headless=False, args=["--disable-dev-shm-usage"])
    context = await browser.new_context(viewport={"width": 1000, "height": 620},
                                        accept_downloads=False)
    page = await context.new_page()
page.set_default_timeout(10000)
print(f"Playwright {version('playwright')} ｜ Chromium {browser.version} ｜ 可视化浏览器已启动")

Playwright 1.62.0 ｜ Chromium 151.0.7922.34 ｜ 可视化浏览器已启动


### 单元2：打开邮箱并等待扫码

In [15]:
from urllib.parse import urlsplit
import re

def inbox_route(url):
    path = urlsplit(url)
    if path.hostname == "mail.qq.com" and re.match(r"^/list/(?:$|[?])", path.fragment):
        return True
    return bool(re.search(r"(?:inbox|folderid=1(?:&|$)|folderid/1(?:/|$))", path.path + "?" + path.query + "#" + path.fragment, re.I))

async def find_inbox():
    deadline = asyncio.get_running_loop().time() + 600
    while asyncio.get_running_loop().time() < deadline:
        for frame in page.frames:
            links = frame.locator(".sidebar-menu-text").filter(has_text=re.compile(r"^\s*(收件箱|Inbox)\s*$", re.I))
            for node in await links.all():
                if await node.is_visible():
                    await node.click()
                    return frame, node
            rows = frame.locator(".mail-list-page-item[data-mailid]")
            if inbox_route(frame.url) and await rows.count() and await rows.first.is_visible():
                return frame, None
        await asyncio.sleep(0.5)
    raise TimeoutError("等待收件箱超时，请确认手机端已同意登录。")

if urlsplit(page.url).hostname != "mail.qq.com":
    await page.goto("https://mail.qq.com/", wait_until="domcontentloaded")
print("请在课程可视化窗口扫码，最长等待 10 分钟。", flush=True)
view, inbox = await find_inbox()
print("登录成功，已进入收件箱。")

请在课程可视化窗口扫码，最长等待 10 分钟。


登录成功，已进入收件箱。


## 第二阶段：采集10封邮件并保存（第3—4课时）

### 单元3 · 定位邮件行、字段与悬停卡片
以带有邮件 ID 的列表行为起点，在当前行内取发件人、主题和时间。字段配置按当前邮箱页面核对后运行。

In [16]:
import json

ROW_CSS = ".mail-list-page-item[data-mailid]"
AD_CSS = ".gg-unread-count"
SENDER_CSS = ".cmp-account-nick"
SUBJECT_CSS = ".mail-subject"
TIME_CSS = ".mail-time"
HOVER_CSS = ".cmp-account-nick"
EMAIL_CSS = ".cmp-email-address"
SCROLL_CSS = ".mail-list-page .ui-float-scroll-body"
EMAIL_RE = re.compile(r"[a-zA-Z0-9.!#$%&'*+/=?^_`{|}~-]+@[a-zA-Z0-9-]+(?:\.[a-zA-Z0-9-]+)+")

async def find_list_view():
    for frame in page.frames:
        if await frame.locator(ROW_CSS).count():
            return frame
    await view.locator(ROW_CSS).first.wait_for(state="attached")
    return view

view = await find_list_view()
print("收件箱邮件行已定位。")

收件箱邮件行已定位。


### 单元4：定义列表解析函数

In [17]:
# 每次由 ID 重新定位，列表重绘后不复用旧的元素句柄。
def get_row(mail_id):
    escaped = json.dumps(mail_id, ensure_ascii=False)
    return view.locator(f"{ROW_CSS}[data-mailid={escaped}]").first

async def visible_node(scope, selector, label, timeout=8):
    until = asyncio.get_running_loop().time() + timeout
    loc = scope.locator(selector)
    while True:
        for node in await loc.all():
            if await node.is_visible():
                return node
        if asyncio.get_running_loop().time() >= until:
            raise ValueError(f"当前页面未找到可见的{label}，请核对单元3或10的选择器。")
        await asyncio.sleep(0.12)

# 同一次浏览器调用读出位置与实际相交的行，兼容页面只渲染部分邮件的列表。
LIST_STATE = """(row, cfg) => {
  let box = cfg.box && document.querySelector(cfg.box);
  if (!box) {
    for (let p=row.parentElement; p; p=p.parentElement) {
      if (/(auto|scroll|overlay)/.test(getComputedStyle(p).overflowY)) {
        box = p; if (p.scrollHeight > p.clientHeight + 1) break;
      }
    }
  }
  box = box || document.scrollingElement;
  if (cfg.top !== null) box.scrollTop = cfg.top;
  const bounds = box.getBoundingClientRect();
  const low = Math.max(0, bounds.top), high = Math.min(innerHeight, bounds.bottom);
  const ids = [...box.querySelectorAll(cfg.rows)].filter(el => {
    const r = el.getBoundingClientRect();
    return r.width > 0 && r.height > 0 && r.bottom > low + 1 && r.top < high - 1;
  }).map(el => el.getAttribute("data-mailid")).filter(Boolean);
  return {top: box.scrollTop, height: Math.min(box.clientHeight,high-low),
          end: Math.max(0,box.scrollHeight-box.clientHeight), ids:[...new Set(ids)]};
}"""

async def snapshot(top=None):
    state = await view.locator(ROW_CSS).first.evaluate(
        LIST_STATE, {"top": top, "box": SCROLL_CSS, "rows": ROW_CSS})
    return state

async def restore(top):
    await snapshot(top)
    # 等待两次绘制，让虚拟列表提交本次位置对应的行。
    await view.evaluate("() => new Promise(r => requestAnimationFrame(() => requestAnimationFrame(r)))")
    return await snapshot()

async def advance(state):
    target = min(state["top"] + max(1, state["height"] - 8), state["end"])
    if target <= state["top"] + 1:
        return None
    new = await restore(target)
    for _ in range(40):
        if new["ids"] and new["ids"] != state["ids"]:
            return new
        await asyncio.sleep(0.1)
        new = await snapshot()
    raise RuntimeError("已滚动，但可见邮件未更新。请核对滚动容器。")

async def read_list_mail(mail_id):
    row = get_row(mail_id)
    if await row.locator(AD_CSS).count():
        return None
    sender_node = await visible_node(row, SENDER_CSS, "发件人")
    subject_node = await visible_node(row, SUBJECT_CSS, "主题")
    time_node = await visible_node(row, TIME_CSS, "时间")
    sender, subject, when = [v.strip() for v in await asyncio.gather(
        sender_node.inner_text(), subject_node.inner_text(), time_node.inner_text())]
    await time_node.hover()
    cards = view.locator(EMAIL_CSS)
    for old in await cards.all():
        await old.wait_for(state="hidden")
    trigger = await visible_node(row, HOVER_CSS, "发件人悬停区")
    await trigger.scroll_into_view_if_needed()
    await view.locator(".mail-list-page .ui-float-scroll-body-scrolling").first.wait_for(state="hidden")
    await trigger.hover()
    card = await visible_node(view, EMAIL_CSS, "邮箱卡片")
    options = [await card.inner_text(), await card.get_attribute("title"),
               await card.get_attribute("href")]
    address = next((m.group() for text in options if text and (m := EMAIL_RE.search(text))), None)
    if address is None:
        raise ValueError("已展开当前发件人卡片，尚未读取到邮箱地址。")
    return {"邮件ID": mail_id, "发件人": sender, "邮箱": address, "主题": subject, "时间": when}

async def open_list():
    if not await view.locator(ROW_CSS).first.is_visible():
        if inbox is not None and await inbox.is_visible():
            await inbox.click()
        else:
            await page.go_back(wait_until="domcontentloaded")
    await view.locator(ROW_CSS).first.wait_for(state="attached")
    return await restore(0)

### 单元5：采集10封

In [18]:
basic_records, basic_positions, seen_basic = [], {}, set()
state = await open_list()
while state and len(basic_records) < 10:
    for mail_id in state["ids"]:
        if mail_id in seen_basic:
            continue
        seen_basic.add(mail_id)
        row = await read_list_mail(mail_id)
        if row is not None:
            basic_records.append(row)
            basic_positions[mail_id] = state["top"]
        if len(basic_records) == 10:
            break
    if len(basic_records) < 10:
        state = await advance(state)
print(f"成功采集 {len(basic_records)} 封普通邮件，目标 10 封。")

成功采集 10 封普通邮件，目标 10 封。


### 单元6：用pandas保存和回读

In [19]:
BASIC_COLUMNS = ["发件人", "邮箱", "主题", "时间"]
def save_records(records, filename, columns):
    path = PRIVATE / filename
    frame = pd.DataFrame(records, columns=columns)
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8-sig")
    temp.replace(path)
    return path

def read_records(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)

basic_df = read_records(save_records(basic_records, "mail_basic.csv", BASIC_COLUMNS))
print(f"CSV 回读 {len(basic_df)} 行 × {len(basic_df.columns)} 列")
basic_checks = []
if len(basic_df):
    for pos in sorted({0, (len(basic_df)-1)//2, len(basic_df)-1}):
        mail_id = basic_records[pos]["邮件ID"]
        await restore(basic_positions[mail_id])
        fresh = await read_list_mail(mail_id)
        matches = [fresh is not None and fresh[col] == basic_df.iloc[pos][col] for col in BASIC_COLUMNS]
        basic_checks.append((pos+1, matches))
        print("第", pos+1, "封，发件人 / 邮箱 / 主题 / 时间：", *matches)
    assert all(all(fields) for _, fields in basic_checks), "网页与 CSV 有差异，请核对。"

CSV 回读 10 行 × 4 列


第 1 封，发件人 / 邮箱 / 主题 / 时间： True True True True


第 5 封，发件人 / 邮箱 / 主题 / 时间： True True True True


第 10 封，发件人 / 邮箱 / 主题 / 时间： True True True True


## 第三阶段：滚动采集与ID去重（第5—6课时）

### 单元7：补全解析和批次循环

In [20]:
async def parse_batch(ids, seen_ids, records, include_details=False, max_mails=0):
    added = 0
    for mail_id in ids:
        if max_mails and len(seen_ids) >= max_mails:
            break
        if mail_id in seen_ids:
            continue
        record = await read_list_mail(mail_id)
        if record is None:
            continue
        if include_details:
            record.update(await read_detail(mail_id))
        records.append(record)
        seen_ids.add(mail_id)
        added += 1
    return added

async def collect_batches(include_details=False, max_mails=0):
    state = await open_list()
    records, seen_ids, positions, log = [], set(), {}, []
    for batch in range(1, 4):
        for mail_id in state["ids"]:
            positions.setdefault(mail_id, state["top"])
        added = await parse_batch(state["ids"], seen_ids, records, include_details, max_mails)
        log.append((batch, added, len(seen_ids)))
        print(f"第 {batch} 批：新增 {added}，累计 {len(seen_ids)}")
        if batch == 3 or (max_mails and len(seen_ids) >= max_mails):
            break
        state = await advance(state)
        if not state:
            print("已到列表末尾。")
            break
    return records, seen_ids, positions, log

### 单元8：执行并保存3批

In [21]:
PAGE_COLUMNS = ["邮件ID"] + BASIC_COLUMNS
page_records, seen_ids, page_positions, page_log = await collect_batches()
page_path = save_records(page_records, "mail_pages.csv", PAGE_COLUMNS)
page_df = read_records(page_path)
print(f"CSV {len(page_df)} 行 × {len(page_df.columns)} 列 ｜ 唯一 ID {page_df['邮件ID'].nunique()} ｜ 重复 {page_df['邮件ID'].duplicated().sum()}")
assert len(page_df) == len(seen_ids) == page_df["邮件ID"].nunique()

第 1 批：新增 13，累计 13


第 2 批：新增 17，累计 30


第 3 批：新增 17，累计 47
CSV 47 行 × 5 列 ｜ 唯一 ID 47 ｜ 重复 0


### 单元9：重复采集验证

In [22]:
assert page_records, "需要至少一条已采集记录。"
replay_ids = [r["邮件ID"] for r in page_records[:5]]
before = page_path.read_bytes()
repeated = await parse_batch(replay_ids, seen_ids, page_records)
save_records(page_records, "mail_pages.csv", PAGE_COLUMNS)
unchanged = before == page_path.read_bytes()
print(f"重复检查 {len(replay_ids)} 封 ｜ 新增 {repeated} ｜ CSV 内容未变 {unchanged}")
assert repeated == 0 and unchanged

重复检查 5 封 ｜ 新增 0 ｜ CSV 内容未变 True


## 第四阶段：详情正文（第7—8课时）

### 单元10 · 进入详情、读取正文、恢复列表
优先读取邮件正文专用框架。若页面使用普通元素，读取指定正文节点。返回后恢复滚动位置并按同一邮件 ID 重新定位。

In [29]:
BODY_CSS = ".mail-detail-content > .qmbox"
BODY_FRAME_CSS = ".mail-detail-content > iframe, .mail-detail-content > .qmbox iframe"
BACK_TEXT = re.compile(r"^(返回|Back)$", re.I)
BACK_CSS = ".xmail-ui-btn:has(+ .back-divider)"
DOWNLOAD_ATTACHMENTS = False

async def body_text():
    deadline = asyncio.get_running_loop().time() + 10
    empty_since = None
    while asyncio.get_running_loop().time() < deadline:
        for host in await view.locator(BODY_FRAME_CSS).all():
            if not await host.is_visible():
                continue
            handle = await host.element_handle()
            frame = await handle.content_frame() if handle else None
            if frame and frame.url != "about:blank":
                node = frame.locator("body")
                await node.wait_for(state="attached")
                await frame.wait_for_load_state("domcontentloaded")
                return (await node.inner_text()).strip()
            if frame and await frame.locator("body").count():
                has_content = await frame.locator("body").evaluate("el => el.childNodes.length > 0")
                if has_content:
                    return (await frame.locator("body").inner_text()).strip()
        for node in await view.locator(BODY_CSS).all():
            if await node.is_visible() and not await node.locator("iframe").count():
                return (await node.inner_text()).strip()
        # 部分邮件有可见详情宿主，已挂载的空正文节点自身高度为 0。
        # 等待短暂稳定期，并在每次检查时排除正在加载的详情与子框架。
        empty_now = await view.locator(".mail-detail-content > .qmbox").evaluate_all("""
            nodes => nodes.some(node => {
                const host = node.parentElement;
                const rect = host.getBoundingClientRect();
                return rect.height > 0 && rect.width > 0 && !host.closest('[aria-busy="true"]')
                    && !host.querySelector('[aria-busy="true"], [role="progressbar"], iframe')
                    && node.childNodes.length === 0 && node.textContent.trim() === "";
            })
        """)
        now = asyncio.get_running_loop().time()
        if empty_now:
            if empty_since is None:
                empty_since = now
            elif now - empty_since >= 0.8:
                return ""
        else:
            empty_since = None
        await asyncio.sleep(0.15)
    raise ValueError("未能确认详情正文已加载，请核对单元10的正文定位。")

async def read_detail(mail_id):
    top = (await snapshot())["top"]
    subject = await visible_node(get_row(mail_id), SUBJECT_CSS, "主题")
    await subject.click()
    try:
        body = await body_text()
        return {"正文": body, "附件文件": ""}
    finally:
        back = view.locator(BACK_CSS)
        if not await back.count():
            back = view.get_by_text(BACK_TEXT)
        for control in await back.all():
            if await control.is_visible():
                await control.click()
                break
        else:
            if inbox is not None and await inbox.is_visible():
                await inbox.click()
            else:
                await page.go_back(wait_until="domcontentloaded")
        await view.locator(ROW_CSS).first.wait_for(state="attached")
        await restore(top)
        await get_row(mail_id).wait_for(state="attached")

### 单元11：先试3封

In [30]:
DETAIL_COLUMNS = PAGE_COLUMNS + ["正文", "附件文件"]
trial_records, _, trial_positions, _ = await collect_batches(include_details=True, max_mails=3)
trial_df = read_records(save_records(trial_records, "mail_details_trial.csv", DETAIL_COLUMNS))
trial_checks = []
for pos, item in trial_df.iterrows():
    mail_id = item["邮件ID"]
    await restore(trial_positions[mail_id])
    fresh = await read_detail(mail_id)
    same = fresh["正文"] == item["正文"]
    restored = await get_row(mail_id).is_visible()
    trial_checks.append((pos+1, len(fresh["正文"]), same, restored))
    print(f"样本 {pos+1} ｜ 正文 {len(fresh['正文'])} 字符 ｜ 与 CSV 一致 {same} ｜ 已返回 {restored}")
assert all(match and back for _, _, match, back in trial_checks)

第 1 批：新增 3，累计 3


样本 1 ｜ 正文 1180 字符 ｜ 与 CSV 一致 True ｜ 已返回 True


样本 2 ｜ 正文 427 字符 ｜ 与 CSV 一致 True ｜ 已返回 True


样本 3 ｜ 正文 464 字符 ｜ 与 CSV 一致 True ｜ 已返回 True


### 单元12：正式采集3批正文

In [31]:
from desensitize import export_public

detail_records, detail_ids, detail_positions, detail_log = await collect_batches(include_details=True)
detail_df = read_records(save_records(detail_records, "mail_details.csv", DETAIL_COLUMNS))
lengths = detail_df["正文"].str.len()
print(f"正文 CSV {len(detail_df)} 行 × {len(detail_df.columns)} 列 ｜ 非空 {int((lengths>0).sum())} ｜ 空正文 {int((lengths==0).sum())}")
print(f"重复 ID {detail_df['邮件ID'].duplicated().sum()} ｜ 与阶段三 ID 集合一致 {detail_ids == seen_ids}")
assert len(detail_df) == detail_df["邮件ID"].nunique()
for name, count in export_public(PRIVATE, PUBLIC).items():
    print(f"{name}：{count} 条")

第 1 批：新增 13，累计 13


第 2 批：新增 17，累计 30


第 3 批：新增 17，累计 47
正文 CSV 47 行 × 7 列 ｜ 非空 45 ｜ 空正文 2
重复 ID 0 ｜ 与阶段三 ID 集合一致 True


mail_basic_public.csv：10 条
mail_pages_public.csv：47 条
mail_details_public.csv：47 条


### 单元13 · 选做附件下载
本次仅采集必做的正文，附件列留空。

In [32]:
print("附件下载未选做。")

附件下载未选做。


### 公开样本核对
读取已经脱敏的基础表，显示第 1、5、10 封的公开代号和对应的逐项核对结果。

In [33]:
public_basic = read_records(PUBLIC / "mail_basic_public.csv")
for position, checks in basic_checks:
    sample = public_basic.iloc[position - 1]
    print(f"第 {position} 封 ｜ 网页与原始 CSV 四项相符 {all(checks)}")
    print(f"  {sample['发件人']}    {sample['邮箱']}")
    print(f"  {sample['主题']}    {sample['时间']}")


第 1 封 ｜ 网页与原始 CSV 四项相符 True
  发件人-16ffbed43396b9e7f284    邮箱-d22235683d4979023443
  主题-bedb6fda4df877c0f06e    时间-358ad33fe1096f4cfe88
第 5 封 ｜ 网页与原始 CSV 四项相符 True
  发件人-c5c05f90c46178863337    邮箱-bc81ad063adcf0e537cf
  主题-61f07949315696ce1694    时间-3dd2c00dcf65b38e1140
第 10 封 ｜ 网页与原始 CSV 四项相符 True
  发件人-6a96a0a8e4ff82085e9e    邮箱-98f30c415e5cef75c554
  主题-f1d80d987d93d0831ba0    时间-60b820f15dfda69ce5e0


### 网页字段定位核对
在实际收件箱中读取第一封普通邮件的元素标签和类名。

In [34]:
await open_list()
row = view.locator(ROW_CSS).filter(has_not=view.locator(AD_CSS)).first
for label, node in [("邮件行", row), ("发件人", row.locator(SENDER_CSS).first),
                    ("主题", row.locator(SUBJECT_CSS).first), ("时间", row.locator(TIME_CSS).first)]:
    tag = await node.evaluate("el => el.tagName.toLowerCase() + '.' + [...el.classList].join('.')")
    print(label, tag)
print("邮件行包含 data-mailid 属性", await row.get_attribute("data-mailid") is not None)


邮件行 div.mail-list-page-item.mail-list-item-compact.mail-list-page-wide-item
发件人 span.cmp-account-nick
主题 span.mail-subject
时间 div.mail-time
邮件行包含 data-mailid 属性 True


## 结束：关闭浏览器

> 下课前保存 Notebook，再执行下面的关闭，最后运行 `stop-course-visual.bat`。

In [37]:
await browser.close()
await p.stop()